# Starfysh: Spatial Deconvolution with Histology Integration

This tutorial demonstrates how to use Starfysh for spatial transcriptomics deconvolution.

Starfysh deconvolves spatial transcriptomics spots into cell type proportions using reference single-cell data. It can optionally integrate histology images for improved accuracy.

## Key Features:
- Cell type deconvolution of spatial spots
- Archetypal factor-based representation
- Optional histology integration
- Expression reconstruction

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

# Import spatialvi
import spatialvi
from spatialvi.external import Starfysh

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## 1. Load Data

We need two datasets:
1. **Spatial transcriptomics data** - the spots/locations to deconvolve
2. **Reference single-cell data** - with cell type annotations

In [ ]:
# Load example spatial data
adata_spatial = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata_spatial.var_names_make_unique()

print("Spatial data:")
print(adata_spatial)

In [ ]:
# For this example, we'll create a synthetic reference
# In practice, you would load your scRNA-seq reference dataset

# Create synthetic reference data
n_cells_per_type = 200
cell_types = ["B cells", "T cells", "Macrophages", "Dendritic cells", "Fibroblasts"]

# Sample cells from spatial data to create synthetic reference
np.random.seed(42)
n_ref_cells = n_cells_per_type * len(cell_types)

# Create reference by subsampling spatial data
indices = np.random.choice(adata_spatial.n_obs, n_ref_cells, replace=True)
adata_ref = adata_spatial[indices].copy()

# Assign synthetic cell type labels
cell_type_labels = np.repeat(cell_types, n_cells_per_type)
np.random.shuffle(cell_type_labels)
adata_ref.obs["cell_type"] = pd.Categorical(cell_type_labels)

print("\nReference data:")
print(adata_ref)
print("\nCell type distribution:")
print(adata_ref.obs["cell_type"].value_counts())

In [ ]:
# Visualize the spatial data
sc.pl.spatial(adata_spatial, color="total_counts", spot_size=100)

## 2. Preprocessing

Preprocess both datasets to ensure they are compatible.

In [ ]:
# Filter genes present in both datasets
sc.pp.filter_genes(adata_spatial, min_cells=10)
sc.pp.filter_genes(adata_ref, min_cells=3)

# Select highly variable genes
sc.pp.highly_variable_genes(adata_spatial, n_top_genes=2000, flavor="seurat_v3")

print(f"Spatial data: {adata_spatial.n_obs} spots, {adata_spatial.n_vars} genes")
print(f"Reference data: {adata_ref.n_obs} cells, {adata_ref.n_vars} genes")

## 3. Initialize Starfysh Model

Create the Starfysh model with spatial and reference data.

In [ ]:
# Initialize Starfysh model
model = Starfysh(
    adata_spatial=adata_spatial,
    adata_ref=adata_ref,
    cell_type_key="cell_type",
    spatial_key="spatial",
    n_factors=5,  # Number of archetypal factors per cell type
    n_hidden=128,  # Hidden layer size
    use_histology=False,  # Set True if histology features available
)

print(f"Number of common genes: {model.n_genes}")
print(f"Number of cell types: {model.n_cell_types}")
print(f"Cell types: {model.cell_types}")

## 4. Train the Model

In [ ]:
# Fit the model
model.fit(
    max_epochs=200,
    lr=1e-3,
    batch_size=256,
    early_stopping=True,
    patience=20,
    device="auto",  # Uses GPU if available
)

print("Model training complete!")

## 5. Get Cell Type Proportions

In [ ]:
# Get estimated cell type proportions
proportions = model.get_proportions(return_dataframe=True)

print("Cell type proportions:")
print(proportions.head(10))

# Summary statistics
print("\nMean proportions per cell type:")
print(proportions.mean())

In [ ]:
# Store proportions in AnnData
model.to_adata()

# Proportions are now in adata_spatial.obsm["starfysh_proportions"]
print("Proportions stored in adata_spatial.obsm['starfysh_proportions']")

## 6. Visualize Results

In [ ]:
# Add proportions to obs for visualization
for ct in model.cell_types:
    adata_spatial.obs[f"prop_{ct}"] = proportions[ct].values

# Visualize cell type proportions spatially
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, ct in enumerate(model.cell_types):
    sc.pl.spatial(
        adata_spatial,
        color=f"prop_{ct}",
        spot_size=80,
        ax=axes[i],
        show=False,
        title=f"{ct} proportion",
    )

# Hide empty subplot
axes[-1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Plot proportion distribution
fig, ax = plt.subplots(figsize=(10, 6))
proportions.boxplot(ax=ax)
ax.set_ylabel("Proportion")
ax.set_xlabel("Cell Type")
ax.set_title("Distribution of Cell Type Proportions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Get Reconstructed Expression

In [ ]:
# Get reconstructed expression
reconstruction = model.get_reconstruction()

print(f"Reconstructed expression shape: {reconstruction.shape}")

# Compare original vs reconstructed for a gene
gene_idx = 0
gene_name = model.common_genes[gene_idx]

X_orig = adata_spatial[:, gene_name].X
if hasattr(X_orig, "toarray"):
    X_orig = X_orig.toarray().flatten()
else:
    X_orig = X_orig.flatten()

plt.figure(figsize=(8, 6))
plt.scatter(X_orig, reconstruction[:, gene_idx], alpha=0.5)
plt.xlabel("Original Expression")
plt.ylabel("Reconstructed Expression")
plt.title(f"Original vs Reconstructed: {gene_name}")

# Add diagonal line
max_val = max(X_orig.max(), reconstruction[:, gene_idx].max())
plt.plot([0, max_val], [0, max_val], "r--", label="y=x")
plt.legend()
plt.show()

## 8. Identify Dominant Cell Types

In [ ]:
# Identify dominant cell type per spot
dominant_ct = proportions.idxmax(axis=1)
adata_spatial.obs["dominant_cell_type"] = dominant_ct.values

# Visualize dominant cell types
sc.pl.spatial(
    adata_spatial,
    color="dominant_cell_type",
    spot_size=80,
    title="Dominant Cell Type per Spot",
)

print("\nDominant cell type counts:")
print(adata_spatial.obs["dominant_cell_type"].value_counts())

## Summary

In this tutorial, we demonstrated:

1. How to prepare spatial and reference single-cell data
2. How to initialize and train the Starfysh model
3. How to extract cell type proportions
4. How to visualize deconvolution results spatially
5. How to identify dominant cell types per spot

Starfysh provides a flexible approach for spatial deconvolution that can optionally incorporate histology information for improved accuracy.